# 01 — SDN DDoS Dataset Generator

**Execution environment:** `wsl -d Wireless`

This notebook creates the master labeled SDN-DDoS dataset. It supports:

- Synthetic/statistical generation for large-scale dataset creation.
- Mininet + Open vSwitch + Ryu/OpenFlow 1.3 integration scaffolding.
- Benign TCP/UDP/ICMP/HTTP traffic classes.
- UDP Flood, ICMP Flood, TCP SYN Flood, HTTP Flood, Slow-rate DDoS.
- UDP amplification-style, Smurf-style, Ping-of-Death-style, LAND, and Packet-In Flood behavioral classes.
- Scenario IDs, attacker count, attack intensity, background traffic.
- Flow, rate, TCP, entropy, flow-table, and controller-oriented features.
- CSV + Parquet export.

The real SDN mode is intended only for an isolated Mininet laboratory. No external targets or third-party reflection services are used.


In [1]:
# WSL/Linux prerequisite check
import platform, sys, shutil
print("OS:", platform.system())
print("Kernel:", platform.release())
print("Python:", sys.version.split()[0])

required = ["mn", "ovs-vsctl", "tcpdump", "iperf3"]
for tool in required:
    print(f"{tool:12} -> {shutil.which(tool)}")


OS: Linux
Kernel: 6.18.33.2-microsoft-standard-WSL2
Python: 3.10.12
mn           -> /usr/bin/mn
ovs-vsctl    -> /usr/bin/ovs-vsctl
tcpdump      -> /usr/bin/tcpdump
iperf3       -> /usr/bin/iperf3


In [2]:
# Python dependencies
# Run once if necessary:
# %pip install -q pandas numpy pyarrow matplotlib

import numpy as np
import pandas as pd
from pathlib import Path
from dataclasses import dataclass, asdict

SEED = 42
rng = np.random.default_rng(SEED)

ROOT = Path.home() / "sdn_ddos_project" / "dataset"
RAW = ROOT / "raw"
PROCESSED = ROOT / "processed"
METADATA = ROOT / "metadata"
SPLITS = ROOT / "splits"

for p in [RAW, PROCESSED, METADATA, SPLITS]:
    p.mkdir(parents=True, exist_ok=True)

WINDOW_S = 5


In [3]:
ATTACK_TYPES = [
    "BENIGN_TCP", "BENIGN_UDP", "BENIGN_ICMP", "BENIGN_HTTP",
    "UDP_FLOOD", "ICMP_FLOOD", "TCP_SYN_FLOOD", "HTTP_FLOOD",
    "SLOW_RATE_DDOS", "UDP_AMPLIFICATION_STYLE", "SMURF_STYLE",
    "PING_OF_DEATH_STYLE", "LAND", "PACKET_IN_FLOOD"
]

ATTACK_ID = {x: i for i, x in enumerate(ATTACK_TYPES)}

INTENSITY = {"low": 0.55, "medium": 1.0, "high": 1.8}
BACKGROUND = {"low": 0.65, "medium": 1.0, "high": 1.45}

@dataclass
class Scenario:
    scenario_id: str
    attack_type: str
    topology: str
    attacker_count: int
    attack_intensity: str
    background_level: str
    duration_s: int
    seed: int

scenarios = []
sid = 0

for attack in ATTACK_TYPES:
    for intensity in ["low", "medium", "high"]:
        sid += 1
        scenarios.append(
            Scenario(
                f"SCN_{sid:04d}",
                attack,
                "linear_3switch",
                1 if attack.startswith("BENIGN") else 2,
                intensity,
                "medium",
                60,
                SEED + sid,
            )
        )

scenario_df = pd.DataFrame(asdict(x) for x in scenarios)
scenario_df.to_csv(METADATA / "scenario_catalog.csv", index=False)
scenario_df


,scenario_id,attack_type,topology,attacker_count,attack_intensity,background_level,duration_s,seed
0,SCN_0001,BENIGN_TCP,linear_3switch,1,low,medium,60,43
1,SCN_0002,BENIGN_TCP,linear_3switch,1,medium,medium,60,44
2,SCN_0003,BENIGN_TCP,linear_3switch,1,high,medium,60,45
3,SCN_0004,BENIGN_UDP,linear_3switch,1,low,medium,60,46
4,SCN_0005,BENIGN_UDP,linear_3switch,1,medium,medium,60,47
5,SCN_0006,BENIGN_UDP,linear_3switch,1,high,medium,60,48
6,SCN_0007,BENIGN_ICMP,linear_3switch,1,low,medium,60,49
7,SCN_0008,BENIGN_ICMP,linear_3switch,1,medium,medium,60,50
8,SCN_0009,BENIGN_ICMP,linear_3switch,1,high,medium,60,51
9,SCN_0010,BENIGN_HTTP,linear_3switch,1,low,medium,60,52


In [4]:
def ln(median, sigma=.35, multiplier=1.0):
    return max(.01, float(rng.lognormal(np.log(median * multiplier), sigma)))

def normal(mean, std, low, high):
    return float(np.clip(rng.normal(mean, std), low, high))

def beta(a, b):
    return float(rng.beta(a, b))

def behavior(s):
    m = INTENSITY[s.attack_intensity]

    if s.attack_type.startswith("BENIGN"):
        b = dict(
            packet_rate=ln(100, .40),
            avg_packet_size=normal(700,220,80,1500),
            new_flows=ln(5,.4),
            active_flows=ln(20,.3),
            completion=beta(8,2),
            request_rate=ln(8,.4),
            source_entropy=normal(.82,.08,.25,1),
            table_growth=ln(2,.35),
            controller_load=normal(.12,.04,.01,.35),
            syn=ln(8,.35), ack=ln(12,.35), rst=ln(1.5,.45)
        )
        if s.attack_type == "BENIGN_HTTP":
            b["request_rate"] *= 1.5
            b["active_flows"] *= 1.3
        elif s.attack_type == "BENIGN_ICMP":
            b["avg_packet_size"] *= .45
        elif s.attack_type == "BENIGN_UDP":
            b["avg_packet_size"] *= .65
        return b

    profiles = {
        "UDP_FLOOD": (1800,420,90,250,.38,.65,.55),
        "ICMP_FLOOD": (2200,256,65,180,.30,.55,.55),
        "TCP_SYN_FLOOD": (1400,64,220,500,.42,.85,.85),
        "HTTP_FLOOD": (700,850,80,220,.68,.62,.62),
        "SLOW_RATE_DDOS": (25,120,130,420,.72,.78,.78),
        "UDP_AMPLIFICATION_STYLE": (3500,950,35,120,.52,.72,.72),
        "SMURF_STYLE": (2600,500,100,300,.25,.68,.68),
        "PING_OF_DEATH_STYLE": (900,1450,20,80,.55,.42,.42),
        "LAND": (900,80,100,160,.02,.60,.60),
        "PACKET_IN_FLOOD": (500,120,700,800,.35,.92,.92),
    }

    pr, aps, nf, af, ent, load, _ = profiles[s.attack_type]

    return dict(
        packet_rate=ln(pr,.45,m),
        avg_packet_size=normal(aps, max(10,aps*.2),40,1500),
        new_flows=ln(nf,.45,m),
        active_flows=ln(af,.4,m),
        completion=beta(1,20) if s.attack_type=="TCP_SYN_FLOOD" else beta(2,8),
        request_rate=ln(max(10,nf*2),.45,m),
        source_entropy=normal(ent,.12,.01,1),
        table_growth=ln(max(10,nf/2),.45,m),
        controller_load=normal(load,.10,.05,1),
        syn=ln(900,.4,m) if s.attack_type=="TCP_SYN_FLOOD" else (ln(100,.4,m) if s.attack_type=="SLOW_RATE_DDOS" else 0),
        ack=ln(180,.4,m) if s.attack_type=="HTTP_FLOOD" else ln(8,.5) if "TCP" in s.attack_type else 0,
        rst=ln(55,.5,m) if "FLOOD" in s.attack_type else ln(10,.5,m),
    )


In [5]:
def protocol_for(a):
    if any(x in a for x in ["ICMP","PING","SMURF"]):
        return "ICMP"
    if "UDP" in a:
        return "UDP"
    return "TCP"

def generate_row(s, ts):
    b = behavior(s)
    protocol = protocol_for(s.attack_type)

    src_n = int(rng.integers(1,49))
    dst_n = int(rng.integers(1,49))
    src_ip = f"10.0.0.{src_n}"
    dst_ip = src_ip if s.attack_type == "LAND" else f"10.0.0.{dst_n}"

    packets = max(1, int(b["packet_rate"] * WINDOW_S))
    bytes_ = max(packets, packets * b["avg_packet_size"])
    new_flows = max(1, int(b["new_flows"]))
    active_flows = max(new_flows, int(b["active_flows"]))
    label = 0 if s.attack_type.startswith("BENIGN") else 1

    return {
        "scenario_id": s.scenario_id,
        "experiment_id": f"EXP_{s.scenario_id}",
        "timestamp": ts,
        "window_start": ts,
        "window_end": ts + pd.Timedelta(seconds=WINDOW_S),
        "window_duration_s": WINDOW_S,
        "topology": s.topology,
        "switch_id": f"s{int(rng.integers(1,4))}",
        "input_port": int(rng.integers(1,5)),
        "output_port": int(rng.integers(1,5)),
        "src_ip": src_ip,
        "dst_ip": dst_ip,
        "src_port": int(rng.integers(1024,65535)),
        "dst_port": int(rng.choice([22,53,80,443,8080])) if protocol != "ICMP" else 0,
        "protocol": protocol,
        "packet_count": packets,
        "byte_count": round(bytes_,2),
        "packet_rate": round(b["packet_rate"],4),
        "byte_rate": round(bytes_/WINDOW_S,4),
        "avg_packet_size": round(b["avg_packet_size"],4),
        "flow_duration_s": round(normal(4.0 if label else 2.8,.8,.2,WINDOW_S),4),
        "new_flows": new_flows,
        "active_flows": active_flows,
        "flow_rate": round(new_flows/WINDOW_S,4),
        "flow_churn": round(new_flows/max(active_flows,1),5),
        "tcp_syn_count": int(max(0,b["syn"])),
        "tcp_ack_count": int(max(0,b["ack"])),
        "tcp_rst_count": int(max(0,b["rst"])),
        "tcp_fin_count": int(max(0,b["ack"]*.2 if protocol=="TCP" else 0)),
        "connection_attempts": new_flows,
        "completed_connections": int(new_flows*b["completion"]),
        "tcp_connection_completion_ratio": round(b["completion"],5),
        "request_count": int(max(1,b["request_rate"]*WINDOW_S)),
        "request_rate": round(b["request_rate"],4),
        "source_entropy": round(b["source_entropy"],5),
        "destination_entropy": round(normal(.75 if label==0 else .35,.12,.02,1),5),
        "port_entropy": round(normal(.70 if label==0 else .30,.14,.02,1),5),
        "in_packets": packets,
        "out_packets": int(max(1,packets*normal(.8,.2,.05,1.5))),
        "in_bytes": round(bytes_,2),
        "out_bytes": round(max(1,bytes_*normal(.8,.2,.05,1.5)),2),
        "flow_table_entries": int(max(1,active_flows*normal(1,.12,.5,1.5))),
        "flow_table_growth": round(b["table_growth"],4),
        "packet_in_count": int(max(0,b["new_flows"]*normal(1.8 if s.attack_type=="PACKET_IN_FLOOD" else .3,.25,0,3))),
        "flow_mod_count": int(max(0,b["new_flows"]*normal(.2,.15,0,2))),
        "controller_load_indicator": round(b["controller_load"],5),
        "attacker_count": s.attacker_count,
        "attack_intensity": s.attack_intensity,
        "background_traffic_level": s.background_level,
        "attack_type": s.attack_type,
        "label": label,
        "label_id": ATTACK_ID[s.attack_type],
    }


In [6]:
# Generate a substantial first dataset.
# Increase to 500/1000 after the schema has been validated.

WINDOWS_PER_SCENARIO = 300
base = pd.Timestamp("2026-01-01", tz="UTC")

rows = []
for i, s in enumerate(scenarios):
    start = base + pd.Timedelta(days=i)
    for w in range(WINDOWS_PER_SCENARIO):
        rows.append(generate_row(s, start + pd.Timedelta(seconds=w*WINDOW_S)))

df = pd.DataFrame(rows).sample(frac=1, random_state=SEED).reset_index(drop=True)

print("Shape:", df.shape)
display(df["attack_type"].value_counts())
display(df.head())


Shape: (12600, 52)


attack_type
LAND                       900
HTTP_FLOOD                 900
BENIGN_UDP                 900
BENIGN_ICMP                900
UDP_FLOOD                  900
BENIGN_HTTP                900
BENIGN_TCP                 900
SLOW_RATE_DDOS             900
PACKET_IN_FLOOD            900
PING_OF_DEATH_STYLE        900
UDP_AMPLIFICATION_STYLE    900
TCP_SYN_FLOOD              900
ICMP_FLOOD                 900
SMURF_STYLE                900
Name: count, dtype: int64

,scenario_id,experiment_id,timestamp,window_start,window_end,window_duration_s,topology,switch_id,input_port,output_port,...,flow_table_growth,packet_in_count,flow_mod_count,controller_load_indicator,attacker_count,attack_intensity,background_traffic_level,attack_type,label,label_id
0,SCN_0038,EXP_SCN_0038,2026-02-07 00:23:45+00:00,2026-02-07 00:23:45+00:00,2026-02-07 00:23:50+00:00,5,linear_3switch,s3,1,2,...,61.5636,6,17,0.68159,2,medium,medium,LAND,1,12
1,SCN_0023,EXP_SCN_0023,2026-01-23 00:22:10+00:00,2026-01-23 00:22:10+00:00,2026-01-23 00:22:15+00:00,5,linear_3switch,s2,3,4,...,37.2822,23,10,0.86711,2,medium,medium,HTTP_FLOOD,1,7
2,SCN_0005,EXP_SCN_0005,2026-01-05 00:10:15+00:00,2026-01-05 00:10:15+00:00,2026-01-05 00:10:20+00:00,5,linear_3switch,s2,1,1,...,1.3194,1,1,0.15187,1,medium,medium,BENIGN_UDP,0,1
3,SCN_0008,EXP_SCN_0008,2026-01-08 00:12:40+00:00,2026-01-08 00:12:40+00:00,2026-01-08 00:12:45+00:00,5,linear_3switch,s2,2,2,...,1.9186,1,0,0.04930,1,medium,medium,BENIGN_ICMP,0,2
4,SCN_0038,EXP_SCN_0038,2026-02-07 00:14:45+00:00,2026-02-07 00:14:45+00:00,2026-02-07 00:14:50+00:00,5,linear_3switch,s2,4,1,...,43.9456,37,44,0.71903,2,medium,medium,LAND,1,12


In [7]:
# Validate and export
assert df["packet_count"].ge(0).all()
assert df["byte_count"].ge(0).all()
assert df["packet_rate"].ge(0).all()
assert df["label"].isin([0,1]).all()
assert df["scenario_id"].notna().all()

csv_path = PROCESSED / "master_sdn_ddos_dataset.csv"
parquet_path = PROCESSED / "master_sdn_ddos_dataset.parquet"

df.to_csv(csv_path, index=False)
df.to_parquet(parquet_path, index=False)

print("CSV:", csv_path)
print("Parquet:", parquet_path)


CSV: /home/satya/sdn_ddos_project/dataset/processed/master_sdn_ddos_dataset.csv
Parquet: /home/satya/sdn_ddos_project/dataset/processed/master_sdn_ddos_dataset.parquet


## Real Mininet/OVS/Ryu mode

The generated dataset above is the statistical development dataset.

For measured data, run the following components in `wsl -d Wireless`:

```text
Mininet
  ↓
OVS
  ↓
OpenFlow 1.3
  ↓
Ryu
  ↓
flow/port/controller telemetry
  ↓
canonical schema
  ↓
Parquet
```

The canonical schema intentionally matches the synthetic dataset so the second notebook can consume either source.


In [8]:
# Save a real-lab topology template for WSL.
topology = r'''
from mininet.topo import Topo

class SDNDDOSTopo(Topo):
    def build(self):
        s1 = self.addSwitch("s1", protocols="OpenFlow13")
        s2 = self.addSwitch("s2", protocols="OpenFlow13")
        s3 = self.addSwitch("s3", protocols="OpenFlow13")

        for i in range(1, 5):
            self.addHost(f"h{i}", ip=f"10.0.0.{i}/24")
            self.addLink(f"h{i}", s1)

        for i in range(5, 9):
            self.addHost(f"h{i}", ip=f"10.0.0.{i}/24")
            self.addLink(f"h{i}", s2)

        for i in range(9, 13):
            self.addHost(f"h{i}", ip=f"10.0.0.{i}/24")
            self.addLink(f"h{i}", s3)

        self.addLink(s1, s2)
        self.addLink(s2, s3)

topos = {"sdnddos": SDNDDOSTopo}
'''

p = METADATA / "sdn_ddos_topology.py"
p.write_text(topology, encoding="utf-8")
print(p)


/home/satya/sdn_ddos_project/dataset/metadata/sdn_ddos_topology.py
